# [1교시]

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
os.getenv('OPENAI_API_KEY')[-10:]

'Vad7SEhRIA'

## ReAct와 LangGraph의 연결 방식

ReAct는 Reason과 Act를 번갈아 수행하는 패턴이다. 질문을 받으면 먼저 현재 답변에 필요한 정보를 판단하고, 부족하면 도구를 호출한다. 그 뒤 도구 결과를 반영해 다시 생각한다.

아래 코드는 이 흐름을 `think -> act -> answer`의 순서로 분해한다. 상태에는 질문, 생각 메모, 도구 결과, 최종 답변만 둬서 흐름이 어떻게 이어지는지 쉽게 보이도록 만든다.

In [2]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START, END

class ReActState(TypedDict):
    question:str
    thought:str
    tool_result:str
    answer:str

def think(state:ReActState):
    question = state['question']
    if '문서' in question or '근거' in question:
        thought = '검색도구가 필요합니다..'
    else:
        thought = '바로 답할수 있습니다.'
    return {'thought':thought}

def act(state:ReActState):
    question = state['question']
    if '문서' in question or '근거' in question:
        tool_result = '벡터DB에서 관련 문단 2개를 찾았습니다.'
    else:
        tool_result = '외부도구가 필요하지 않다.'
    return {'tool_result':tool_result}
def answer(state:ReActState):
    answer = f"질문:{state['question']}\n생각:{state['thought']}\n도구:{state['tool_result']}"
    return {'answer':answer}

workflow = StateGraph(ReActState)
workflow.add_node('think',think)
workflow.add_node('act',act)
workflow.add_node('answer',answer)
workflow.add_edge(START,'think')
workflow.add_edge('think','act')
workflow.add_edge('act','answer')
workflow.add_edge('answer',END)
app = workflow.compile()
kwargs = {
    'question':'문서 근거가 필요한 더미 질문입니다.',
    'thought':'',
    'tool_result':'',
    'answer':''
}
result = app.invoke(kwargs)
print(result['answer'])

질문:문서 근거가 필요한 더미 질문입니다.
생각:검색도구가 필요합니다..
도구:벡터DB에서 관련 문단 2개를 찾았습니다.


# [2교시]

In [3]:
import chromadb
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START,END
from openai import OpenAI
import chromadb
import os
import re
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

embedding_function = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
chroma_client = chromadb.PersistentClient(path='chroma_lesson_01_v3')
collection = chroma_client.get_or_create_collection(
    name = 'react_rag',
    embedding_function=embedding_function
)
if collection.count() == 0:
    collection.add(
        ids=["doc_langgraph", "doc_react", "doc_rag", "doc_chroma"],
        documents=[
            "LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.",
            "ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.",
            "RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.",
            "ChromaDB는 문서 임베딩을 저장하고 유사한 문서를 검색하는 벡터DB다.",
        ],
    )    

# 그래프에 들어갈 노드 함수구성 --> Tool
class ReActRAGState(TypedDict):
    question:str
    thought:str
    action:str
    tool_result:str
    answer:str

def think(state:ReActRAGState):
    question = state['question'].lower()
    if any(keyword in question for keyword in ['근거','문서','설명','rag']):
        thought = 'CromaDB에서 근거를 먼저 찾아야 한다'
        action = 'search'
    else:
        thought = '바로 답할 수 있다'
        action = 'respond'
    return {'thought':thought,'action':action}


def search_context(state:ReActRAGState):# 내부문서
    result = collection.query(query_texts=[state['question']], n_results=2)
    context = '\n'.join(result['documents'][0])
    return {'tool_result':context}

client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
def respond(state:ReActRAGState):  # LLM이 답변
    answer = f"질문:{state['question']}\n생각:{state['thought']}\n도구:{state['tool_result']}\n\n답변을 생성하세요"
    response = client.chat.completions.create(
        model = 'gpt-5.4-nano',
        messages=[
            {'role':'system','content':'당신은 인공지능 자연어 NLP 전문가입니다.'},
            {'role':'user','content':answer}
        ],
        temperature=0
    )
    return {'answer':response.choices[0].message.content.strip()}

# graph구성
workflow = StateGraph(ReActRAGState)   
workflow.add_node('think',think)
workflow.add_node('search_context',search_context)
workflow.add_node('respond',respond)
workflow.add_edge(START, 'think')
workflow.add_conditional_edges(
    'think',
    lambda state : state['action'],
    {"search":'search_context', "respond":"respond"}
)
workflow.add_edge('search_context','respond')
workflow.add_edge('respond',END)

app = workflow.compile()
kwargs = {
    'question':'LangGraph와 RAG가 연결될때 왜 ReAct는 유용한가?.',
    'thought':'',
    'action':'',
    'tool_result':'',
    'answer':''
}
result = app.invoke(kwargs)
print('route', result['action'])
print('thought', result['thought'])
print('context')
print(result['tool_result'])
print('answer')
print(result['answer'])


c:\miniconda\envs\edu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8583.27it/s]


route search
thought CromaDB에서 근거를 먼저 찾아야 한다
context
LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.
ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.
answer
LangGraph와 RAG를 연결할 때 **ReAct가 유용한 이유**는, RAG가 “근거를 찾아오는 단계”에 강점이 있는 반면 **그 근거를 바탕으로 다음 행동(도구 호출/추가 검색/최종 답변)을 어떻게 선택할지**를 에이전트가 잘 결정해야 하기 때문입니다. ReAct는 이 “결정 과정”을 구조적으로 만들어 줍니다.

- **RAG(예: ChromaDB)에서 근거를 먼저 찾기**
  - 질문에 대해 먼저 관련 문서를 검색해야 합니다.
  - ReAct는 추론(Reasoning) 중에 “지금은 검색이 필요한가?”를 판단하고, 필요하면 **검색 도구를 호출(Acting)** 하도록 자연스럽게 흐름을 만듭니다.
  - 즉, “근거를 먼저 찾아야 한다”는 요구를 에이전트의 행동 정책으로 고정하기 쉽습니다.

- **LangGraph의 상태 기반 다단계 흐름과 잘 맞음**
  - LangGraph는 상태(state)를 중심으로 여러 노드(단계)를 연결해 **다단계 에이전트**를 설계합니다.
  - ReAct는 각 단계에서
    1) 현재까지의 정보로 추론하고  
    2) 다음에 어떤 도구를 쓸지 행동으로 옮기며  
    3) 그 결과를 다시 상태에 반영  
    하는 패턴을 제공합니다.
  - 그래서 LangGraph의 “검색 → 검증/추가검색 → 답변 생성” 같은 그래프를 만들 때, 각 노드가 **ReAct의 추론/행동 전환**을 기준으로 명확해집니다.

- **근거 기반으로 ‘추가 행동’을 반복할 수 있음**
  - RAG 결과가 부족하면 “추가 검색”이나 “다른 쿼리로 재검색” 같은 반복이 필요합니다.
  - ReAct는 reasoning과 acting을 번갈아 수행하므로, 문서가 충분하지 않을 때

# [3교시]

In [4]:
# 기업문서를 벡터Db화  pdf
from pathlib import Path
from typing import Iterable
from uuid import uuid4

import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from pypdf import PdfReader

# 문서경로 지정
enterprise_folder = Path(r'.\documents')
# pdf 파일 읽기
def read_pdf_file(path:Path)->str:
    reader = PdfReader(str(path))
    pages:list[str]=[]

    for page in reader.pages:    
        pages.append(page.extract_text() or "")
    return '\n'.join(pages)
    
enterprise_embedding = SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')
enterprize_client = chromadb.PersistentClient()
enterprise_collection = enterprize_client.get_or_create_collection(
    name='enterprise_document',
    embedding_function=enterprise_embedding,
    metadata={'hnsw:space':'cosine'}
)
# txt, md 파일처럼 일반 파일
def read_text_file(path:Path)->str:
    return path.read_text(encoding='utf-8',errors='ignore')

def chunk_text(text:str, chunk_size:int=900, overlab:int=150)->list[str]:
    cleaned = " ".join(text.split())
    if not cleaned:
        return []
    chunks : list[str] = []
    start = 0
    while start < len(cleaned):
        end = min(start+chunk_size, len(cleaned))
        chunks.append(cleaned[start:end])
        if end >= len(cleaned):
            break
        start = end-overlab
    return chunks

from glob import glob
def iter_documents(folder:Path)->Iterable[tuple[Path,str]]:
    for path in folder.rglob('*'):
        if path.is_file():
            suffix = path.suffix.lower()
            if suffix in {'.txt', ".md"}:
                yield path, read_text_file(path)
            elif suffix == '.pdf':
                yield path, read_pdf_file(path)
        

documents_to_add: list[str] = []
metadatas: list[dict[str, str]] = []
ids: list[str] = []

if not enterprise_folder.exists():
    print(f"경고: 기업 문서 폴더가 없습니다: {enterprise_folder}")
    print("폴더 경로를 확인하거나 폴더를 생성한 뒤 다시 실행하세요.")
else:
    for file_path, raw_text in iter_documents(enterprise_folder):
        for index, chunk in enumerate(chunk_text(raw_text), start=1):
            ids.append(f"{file_path.stem}-{index}-{uuid4().hex[:8]}")
            documents_to_add.append(chunk)
            metadatas.append({
                "source_file": file_path.name,
                "source_type": file_path.suffix.lower().lstrip("."),
                "chunk_index": str(index),
            })

    if documents_to_add:
        enterprise_collection.add(
            ids=ids,
            documents=documents_to_add,
            metadatas=metadatas,
        )
        print(f"기업 문서 {len(ids)}개 청크를 벡터DB에 저장했다.")
    else:
        print("enterprise_documents/ 폴더에서 적재할 문서를 찾지 못했다.")

    print("collection count:", enterprise_collection.count())

기업 문서 43개 청크를 벡터DB에 저장했다.
collection count: 64


In [5]:
test = ''
' '.join(test.split())

''

In [6]:
test = "hello"
test or "None"

'hello'

# [4교시]

In [7]:
# 생성한 벡터db에서 문장이나 단어를 검색해 보세요
from openai import OpenAI
import os
import json
result = enterprise_collection.query(query_texts=['대회 규칙에 대해서 알려주세요'], n_results=2)
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
response = client.chat.completions.create(
        model = 'gpt-5.4-nano',
        messages=[
            {'role':'system','content':'당신은 기업내부 또는 외부문서를 기반으로 답변하는 친절한 페르소나 입니다. \n문서에서 추출한 청크들을 답변해 주세요.'},
            {'role':'user','content':json.dumps(result, ensure_ascii=False)}
        ],
        temperature=0
    )
print(response.choices[0].message.content.strip())

문서(enterprise2.pdf / enterprise.pdf)에서 발췌된 내용 기준으로, **19~21번 조항**은 다음과 같습니다.

## 19. 인터넷/전송 오류 등으로 인한 책임 면책 및 참가 제한
- 대회 스폰서와 데이콘은 **대회 웹 사이트의 시스템 오류**, **기타 통신 전송 오작동**으로 인해 제출물을 전송할 수 없거나,
- **제출물 또는 자료가 손상된 경우** 책임을 지지 않습니다.
- 예시로는 **케이블 연결, 위성 전송, 서버/제공자/컴퓨터 장비의 기술적 고장, 소프트웨어/하드웨어 고장, 네트워크 유실 또는 사용 불가능, 시스템/인적 오류 및 고장, 인터넷 또는 대회 웹 사이트 트래픽 혼잡** 등이 포함되며,
- 이러한 원인(또는 조합)으로 인해 **참가자의 참가가 제한될 수 있음**을 명시합니다.

## 20. 취소/수정/중지 및 실격 권한
- **컴퓨터 바이러스, 버그, 변조, 무단 개입, 사기, 기술적 오류 또는 행정**, 그리고 **보안/공정성/무결성 등에 손상을 주거나 영향을 미치는 기타 원인**으로 인해 대회가 계획대로 실행될 수 없는 경우,
- 대회 스폰서는 **경품 행사를 취소, 종료, 수정 또는 중지**할 권한이 있습니다.
- 또한 **제출 과정이나 대회/대회 웹 사이트의 다른 부분을 조작한 참가자**는 **실격 처리**할 수 있습니다.
- 대회 웹 사이트를 포함해 웹 사이트를 **의도적으로 손상**시키거나, 대회 스폰서와 데이콘의 **합법적 운영을 저해**하려는 시도는 **형법 및 민법 위반**에 해당할 수 있으며,
- 그러한 시도가 이루어지면 대회 스폰서 및 데이콘은 **해당 법률의 최대 범위 내에서 손해배상 청구**를 할 수 있습니다.

## 21. 고용 제안이 아님(계약 아님)
- 대회 웹 사이트에 별도로 명시되지 않는 한,
- 제출물 제출, 상금 수여, 본 규정의 어떤 내용도 **대회 스폰서 또는 대회 주체와의 고용 제안 또는 계약으로 해석되지 않습니다.**
- 참가자는 제출물을 **자발적으로 제출**했으며, 

In [8]:
# 내부문서 기반 RAG (출처 포함)


class EnterpriseRAGState(TypedDict):
    question:str
    retrieved_context:str
    source_items : list[dict[str,str]]
    answer :str
    route:str

def retrieve_with_sources(state:EnterpriseRAGState):
    if enterprise_collection.count == 0:
        return{
            'retrieved_context':'',
            'source_items':[],
            'route':'external'  # 외부문서
        }
    result = enterprise_collection.query(
        query_texts=[state['question']],
        n_results=4,
        include=['documents','metadatas','distances']
    )
    documents = result.get('documents',[[]])[0]
    metadatas = result.get('metadatas',[[]])[0]
    distances = result.get('distances',[[]])[0]
    source_items: list[dict[str, str]] = []
    context_lines: list[str] = []

    for idx, (doc, meta, dist) in enumerate(zip(documents, metadatas, distances), start=1):
        source_id = f"S{idx}"
        file_name = (meta or {}).get("source_file", "unknown")
        chunk_index = (meta or {}).get("chunk_index", "?")
        distance = f"{float(dist):.4f}" if dist is not None else "N/A"

        source_items.append(
            {
                "source_id": source_id,
                "file_name": str(file_name),
                "chunk_index": str(chunk_index),
                "distance": distance,
            }
        )

        context_lines.append(
            f"[{source_id}] file={file_name}, chunk={chunk_index}, distance={distance}\n{doc}"
        )

    if context_lines:
        return {
            "retrieved_context": "\n\n".join(context_lines),
            "source_items": source_items,
            "route": "internal",
        }

    return {
        "retrieved_context": "",
        "source_items": [],
        "route": "external",
    }

def external_reference_dummy(state: EnterpriseRAGState):
    source_items = [
        {
            "source_id": "E1",
            "file_name": "external_policy_reference.md",
            "chunk_index": "1",
            "distance": "N/A",
        },
        {
            "source_id": "E2",
            "file_name": "external_rag_reference.md",
            "chunk_index": "1",
            "distance": "N/A",
        },
    ]
    retrieved_context = (
        "[E1] file=external_policy_reference.md, chunk=1, distance=N/A\n"
        f"질문 '{state['question']}'에 대해 외부 공개 문서의 정책 구조는 일반적으로 목적, 범위, 책임, 절차, 예외 처리로 구성된다.\n\n"
        "[E2] file=external_rag_reference.md, chunk=1, distance=N/A\n"
        "검색된 내부 문서가 없을 때는 외부 공개 자료를 참고해 답변의 뼈대를 만들고, 실제 운영 규정은 반드시 내부 문서로 재검증해야 한다."
    )
    return {
        "retrieved_context": retrieved_context,
        "source_items": source_items,
        "route": "external",
    }


def answer_with_sources(state: EnterpriseRAGState):
    if not state["retrieved_context"].strip():
        return {
            "answer": "검색된 기업 내부 문서가 없어 답변을 생성할 수 없다. 먼저 문서를 벡터DB에 적재해줘.",
        }

    source_labels = ", ".join(item["source_id"] for item in state["source_items"])
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": (
                    "당신은 기업 내부 지식 기반 도우미다. "
                    "반드시 제공된 문맥만 사용해서 답변하고, 핵심 주장 뒤에 출처 라벨을 붙여라. "
                    f"사용 가능한 출처 라벨은 {source_labels} 뿐이다."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"질문: {state['question']}\n\n"
                    f"검색 문맥:\n{state['retrieved_context']}\n\n"
                    "요구사항:\n"
                    "1) 답변 본문에 근거 문장 뒤에 [S1] 같은 출처 라벨을 붙여라.\n"
                    "2) 문맥에 없는 내용은 추측하지 마라."
                ),
            },
        ],
    )
    return {"answer": response.choices[0].message.content.strip()}


enterprise_rag = StateGraph(EnterpriseRAGState)
enterprise_rag.add_node("retrieve_with_sources", retrieve_with_sources)
enterprise_rag.add_node("external_reference_dummy", external_reference_dummy)
enterprise_rag.add_node("answer_with_sources", answer_with_sources)
enterprise_rag.add_edge(START, "retrieve_with_sources")
enterprise_rag.add_conditional_edges(
    "retrieve_with_sources",
    lambda state: state["route"],
    {
        "internal": "answer_with_sources",
        "external": "external_reference_dummy",
    },
)
enterprise_rag.add_edge("external_reference_dummy", "answer_with_sources")
enterprise_rag.add_edge("answer_with_sources", END)
enterprise_rag_app = enterprise_rag.compile()

# question = "우리 회사 문서 기준으로 해당 정책의 핵심 절차를 설명해줘"
question = "내가 판사가 되려면 어떻게 공부해야 돼?"
result = enterprise_rag_app.invoke(
    {
        "question": question,
        "retrieved_context": "",
        "source_items": [],
        "answer": "",
        "route": "",
    }
)

print("question:", question)
print("route:", result.get("route", ""))
print("\nanswer:\n", result["answer"])

print("\n[출처 목록]")
if result["source_items"]:
    for item in result["source_items"]:
        print(
            f"[{item['source_id']}] file={item['file_name']}, chunk={item['chunk_index']}, distance={item['distance']}"
        )
else:
    print("검색된 출처가 없습니다.")

question: 내가 판사가 되려면 어떻게 공부해야 돼?
route: internal

answer:
 제공된 문맥에는 판사가 되기 위한 공부 방법에 대한 정보가 포함되어 있지 않습니다. 따라서 판사가 되기 위해 필요한 공부 방법에 대한 구체적인 조언을 제공할 수 없습니다. 다른 질문이 있으시면 도와드리겠습니다.

[출처 목록]
[S1] file=enterprise2.pdf, chunk=16, distance=0.6068
[S2] file=enterprise2.pdf, chunk=9, distance=0.6193
[S3] file=enterprise2.pdf, chunk=11, distance=0.6211
[S4] file=enterprise.pdf, chunk=3, distance=0.6277


# [5교시]

In [9]:
################################################
# LangGraph StateGraph 상태 설계
################################################
# openai가 질문의도를 읽고 chromadb가 실제 문서를 검색, 검색 결과를 바탕으로 답변을 생성

######## 상태설계를 먼저해야 하는 이유 ##########
# 질문을 정규화 -> 의도를 추론 -> chromaDB에서 실제 근거를 찾고 -> openai가 그 근거를 바탕으로 답변을 생성

import re
from pathlib import Path
from typing_extensions import TypedDict
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)
import os
from sentence_transformers import SentenceTransformer
import chromadb


client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

class StableEmbeddingFunction:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-V2')
    
    def name(self):
        return 'stable_local'        
    
    def __call__(self,input):
        vectors = self.model.encode(
            input,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        return vectors.tolist()

    def embed_query(self,input):
        return self.__call__(input)
    
    
chroma_client = chromadb.PersistentClient()

try:  # 컬렉션은 있으면 생성오류가 발생 그리고 벡터db의 저장할 데이터의 형태가 변경되어도 기존에 존재하면 반영안됨
    chroma_client.delete_collection('test')
except:
    pass

collection = chroma_client.get_or_create_collection(
    name='test',
    embedding_function=StableEmbeddingFunction()
)

# collection 에 데이터 추가
if collection.count() == 0:
    collection.add(
        ids=["doc_langgraph", "doc_react", "doc_rag", "doc_chroma"],
        documents=[
            "LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.",
            "ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.",
            "RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.",
            "ChromaDB는 문서 임베딩을 저장하고 유사한 문서를 검색하는 벡터DB다.",
        ],
    )   

# 노드에 적용할 함수
# 그래프 노드에 추가
# 엣지적용
# 컴파일  app
# app.invoke --> 결과 
class StateDesignState(TypedDict):
    question: str
    normalized_question: str
    query_intent: str
    retrieved_context: str
    draft_answer: str
    final_answer: str


def normalize_question(state: StateDesignState):
    normalized = re.sub(r"\s+", " ", state["question"].strip().lower())
    return {"normalized_question": normalized}


def infer_intent(state: StateDesignState):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content": "질문의 의도를 search, explain, compare 중 하나로만 출력한다.",
            },
            {"role": "user", "content": state["normalized_question"]},
        ],
        temperature=0,
    )
    intent = response.choices[0].message.content.strip().lower()
    if intent not in {"search", "explain", "compare"}:
        text = state["normalized_question"]
        if "비교" in text or "차이" in text:
            intent = "compare"
        elif "설명" in text or "왜" in text or "무엇" in text:
            intent = "explain"
        else:
            intent = "search"
    return {"query_intent": intent}


def retrieve_context(state: StateDesignState):
    result = collection.query(query_texts=[state["normalized_question"]], n_results=2)
    context = "\n".join(result["documents"][0])
    return {"retrieved_context": context}


def draft_answer(state: StateDesignState):
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": "주어진 근거를 바탕으로 3문장 이내의 한국어 답변 초안을 만든다.",
            },
            {
                "role": "user",
                "content": f"질문: {state['question']}\n의도: {state['query_intent']}\n\n근거:\n{state['retrieved_context']}",
            },
        ],
        temperature=0,
    )
    return {"draft_answer": response.choices[0].message.content.strip()}


def finalize_answer(state: StateDesignState):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content": "초안을 정리하고, 마지막에 한 줄로 이유를 덧붙인다.",
            },
            {
                "role": "user",
                "content": f"질문: {state['question']}\n초안: {state['draft_answer']}",
            },
        ],
        temperature=0,
    )
    return {"final_answer": response.choices[0].message.content.strip()}


workflow = StateGraph(StateDesignState)
workflow.add_node("normalize_question", normalize_question)
workflow.add_node("infer_intent", infer_intent)
workflow.add_node("retrieve_context", retrieve_context)
workflow.add_node("draft_answer", draft_answer)
workflow.add_node("finalize_answer", finalize_answer)

workflow.add_edge(START, "normalize_question")
workflow.add_edge("normalize_question", "infer_intent")
workflow.add_edge("infer_intent", "retrieve_context")
workflow.add_edge("retrieve_context", "draft_answer")
workflow.add_edge("draft_answer", "finalize_answer")
workflow.add_edge("finalize_answer", END)

app = workflow.compile()

result = app.invoke(
    {
        "question": "LangGraph와 RAG의 상태 설계가 왜 중요한가?",
        "normalized_question": "",
        "query_intent": "",
        "retrieved_context": "",
        "draft_answer": "",
        "final_answer": "",
    }
)
print("normalized:", result["normalized_question"])
print("intent:", result["query_intent"])
print("context:")
print(result["retrieved_context"])
print("answer:")
print(result["final_answer"])

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11436.79it/s]


normalized: langgraph와 rag의 상태 설계가 왜 중요한가?
intent: explain
context:
LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.
RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.
answer:
LangGraph와 RAG의 상태 설계가 중요한 이유는, 두 시스템 모두 여러 단계의 처리 결과를 다음 단계로 정확히 전달해야 하며 이때 “무엇을 어떤 형태로 상태에 담고, 다음 노드가 어떻게 읽고 갱신하느냐”가 전체 흐름의 일관성과 안정성을 좌우하기 때문입니다. LangGraph에서는 상태가 에이전트의 판단 결과와 분기(라우팅) 조건을 관리하는 중심이 되어, 잘 설계된 상태는 예측 가능한 실행과 안정적인 제어를 가능하게 합니다. RAG에서는 검색된 문서, 근거(출처), 메타데이터, 그리고 생성에 사용된 컨텍스트를 상태로 체계적으로 보관해야 답변 품질이 높아지고, 나중에 어떤 근거로 답했는지 추적·검증하기도 쉬워집니다. 결국 상태 설계의 수준에 따라 정확도, 확장성, 디버깅/운영 효율이 크게 달라집니다.  

이유: 상태를 잘 정의하면 단계 간 데이터 흐름이 명확해져 품질·제어·추적성이 동시에 확보되기 때문입니다.


# [6교시]

In [10]:
############ Router 기반 분기 설계 ##################
# openai로 질문을 분류, 필요할 때만 chromaDB에서 근거를 찾은뒤 답변
# 분류결과에 따라서 필요한 노드만 호출

# 한국어 임베딩 모델
import os
import re
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [11]:
# openai client
client = OpenAI()
# 임베딩 함수  허깅페이스 :This is a sentence-transformers model 찾아라.
st_ef =  embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='jhgan/ko-sroberta-multitask'
)
chroma_client = chromadb.PersistentClient()  #DB
collection = chroma_client.get_or_create_collection(   # table
    name = 'router_colleciton',
    embedding_function=st_ef
)
# collection 에 데이터 추가
if collection.count() == 0:
    collection.add(
        ids=["doc_langgraph", "doc_react", "doc_rag", "doc_chroma"],
        documents=[
            "LangGraph는 상태 기반 그래프로 다단계 에이전트를 설계한다.",
            "ReAct는 reasoning과 acting을 번갈아 수행해 도구 사용을 결합한다.",
            "RAG는 벡터DB 검색 결과를 근거로 답변 품질을 높인다.",
            "ChromaDB는 문서 임베딩을 저장하고 유사한 문서를 검색하는 벡터DB다.",
        ],
    )


class RouterState(TypedDict):
    question:str
    route:str
    retrived_context:str
    answer:str

ALLOWED_ROUTES = {
    'feq','rag','tool'
}
# 라우터 기반 분기
def classify_route(state:RouterState):
    prompt = [
        {'role':'system','content':'질문을 faq, rag,tool중 하나로만 분류한다. 답변은 한 단어로만 출력한다'},
        {'role':'user','content':state['question']}

    ]
    try:
        response = client.chat.completions.create(
            model='gpt-5.4-nano',
            messages=prompt,
            temperature=0
        )
        route = (
            response.choices[0]
            .message.content
            .strip()
            .lower()
        )
    except Exception:
        return "raq"
    if route not in ALLOWED_ROUTES:
        text = state["question"].lower()

        if (
            "계산" in text
            or any(symbol in text for symbol in ["+", "-", "*", "/"])
        ):
            route = "tool"

        elif (
            "문서" in text
            or "근거" in text
            or "rag" in text
        ):
            route = "rag"

        else:
            route = "faq"

    return {
        "route": route
    }

def retrieve_context(state: RouterState):
    result = collection.query(
        query_texts=[state["question"]],
        n_results=2,
    )

    documents = result["documents"][0]

    return {
        "retrieved_context": "\n".join(documents)
    }


def faq_answer(state: RouterState):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "짧고 명확한 한국어 정의를 "
                    "한 문단으로 답한다."
                ),
            },
            {
                "role": "user",
                "content": state["question"],
            },
        ],
        temperature=0,
    )

    return {
        "answer": response.choices[0].message.content.strip()
    }


def rag_answer(state: RouterState):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    "주어진 근거만 사용해 한국어로 답한다."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"질문: {state['question']}\n\n"
                    f"근거:\n{state['retrieved_context']}"
                ),
            },
        ],
        temperature=0,
    )

    return {
        "answer": response.choices[0].message.content.strip()
    }


def tool_answer(state: RouterState):
    expression = re.findall(
        r"[0-9\+\-\*/\(\)\.]+",
        state["question"].replace(" ", "")
    )

    if expression:
        try:
            tool_result = str(
                eval(
                    expression[0],
                    {"__builtins__": {}},
                    {}
                )
            )

        except Exception:
            tool_result = "도구 계산에 실패했다."

    else:
        tool_result = (
            "실행형 도구가 필요한 질문으로 분류되었다."
        )

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    "계산 결과를 짧게 설명하는 "
                    "한국어 답변을 쓴다."
                ),
            },
            {
                "role": "user",
                "content": (
                    f"질문: {state['question']}\n"
                    f"도구 결과: {tool_result}"
                ),
            },
        ],
        temperature=0,
    )

    return {
        "answer": response.choices[0].message.content.strip()
    }


workflow = StateGraph(RouterState)

workflow.add_node("classify_route", classify_route)
workflow.add_node("retrieve_context", retrieve_context)
workflow.add_node("faq_answer", faq_answer)
workflow.add_node("rag_answer", rag_answer)
workflow.add_node("tool_answer",tool_answer)
workflow.add_edge(START,"classify_route")
workflow.add_conditional_edges(
    "classify_route",
    lambda state: state["route"],
    {
        "faq": "faq_answer",
        "rag": "retrieve_context",
        "tool": "tool_answer",
    },
)

workflow.add_edge("retrieve_context","rag_answer")
workflow.add_edge("faq_answer",END)
workflow.add_edge("rag_answer",END)
workflow.add_edge("tool_answer",END)
app = workflow.compile()
result = app.invoke(
    {
        "question": "LangGraph의 정의가 무엇인가?",
        "route": "",
        "retrived_context": "",
        "answer": "",
    }
)
print("route:", result["route"])
print(result["answer"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3986.14it/s]


route: faq
LangGraph는 자연어 처리(NLP)와 그래프 이론을 결합한 모델로, 언어 데이터를 그래프 형태로 표현하여 의미론적 관계를 분석하고 이해하는 데 사용됩니다. 이를 통해 문맥, 의미, 관계 등을 효과적으로 파악할 수 있으며, 다양한 언어 관련 작업에 활용됩니다.


# [7교시]

In [12]:
############################################################################################
# SuperVisor 패턴 : 하나의 상위노드가 여러 작업자를 배치해서 각 작업의 결과를 모아서 최종 답변
############################################################################################
# 초안
# 비평
# 품질 점수
# 80점미만
# 재작성

#       Supervisor
#       Reasearcher
#       Writer
#       Critic
# faile         pass
# Writer        Finalizer

In [13]:
# 슈퍼바이저를 한단계 더 발전시켜서 에이전트 개념확장
#                     Supervisor
#                          │
#         ┌────────────────┼────────────────┐
#         │                │                │
#         ▼                ▼                ▼
#    ResearchAgent    WriterAgent    CriticAgent
#         │                │                │
#         └────────────────┼────────────────┘
#                          ▼
#                     Finalizer

#  Supervisor--> 오케스트레이터

In [16]:
import os
import re
from pathlib import Path
import chromadb
from chromadb.utils import embedding_functions
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START,END
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

# openai client
client = OpenAI()
# 임베딩 함수  허깅페이스 :This is a sentence-transformers model 찾아라.
st_ef =  embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='jhgan/ko-sroberta-multitask'
)
chroma_client = chromadb.PersistentClient()  #DB
collection = chroma_client.get_or_create_collection(   # table
    name = 'Supervisor_MultiAgent',
    embedding_function=st_ef
)

if collection.count() == 0:
    collection.add(
        ids=[
            "doc_langgraph",
            "doc_react",
            "doc_rag",
            "doc_supervisor",
        ],
        documents=[
            "LangGraph는 상태 기반 그래프로 역할별 노드를 연결한다.",
            "ReAct는 생각과 행동을 번갈아 수행하며 중간 도구를 사용할 수 있다.",
            "RAG는 벡터DB에서 찾은 근거를 답변 생성에 사용한다.",
            "Supervisor는 여러 에이전트의 순서와 책임을 조율한다.",
        ],
    )

# --------------------------------------------------
# State
# --------------------------------------------------

class SupervisorState(TypedDict):
    task: str
    research_notes: str
    draft: str
    critique: str
    score: int
    revision_count: int
    final_answer: str

# --------------------------------------------------
# Research Agent
# --------------------------------------------------

def researcher(state: SupervisorState):

    result = collection.query(
        query_texts=[state["task"]],
        n_results=3,
    )

    context = "\n".join(
        result["documents"][0]
    )

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                "당신은 Research Agent이다. "
                "근거를 간결하게 정리하라."
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

근거:
{context}
"""
            }
        ]
    )

    return {
        "research_notes":
        response.choices[0].message.content.strip()
    }


# --------------------------------------------------
# Writer Agent
# --------------------------------------------------

def writer(state: SupervisorState):

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                "당신은 Writer Agent이다."
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

조사내용:
{state['research_notes']}

이전 비평:
{state['critique']}

비평을 반영하여 설명문을 작성하라.
"""
            }
        ]
    )

    return {
        "draft":
        response.choices[0].message.content.strip()
    }


# --------------------------------------------------
# Critic Agent
# --------------------------------------------------

def critic(state: SupervisorState):
    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                """
당신은 Critic Agent이다.

반드시 아래 형식으로 답한다.

SCORE: 숫자

CRITIQUE:
비평 내용
"""
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

초안:
{state['draft']}
"""
            }
        ]
    )

    text = response.choices[0].message.content.strip()

    score = 50

    try:
        import re

        match = re.search(
            r"SCORE:\s*(\d+)",
            text
        )

        if match:
            score = int(match.group(1))

    except Exception:
        pass

    return {
        "critique": text,
        "score": score,
    }


# --------------------------------------------------
# Supervisor
# --------------------------------------------------

def supervisor(state: SupervisorState):
    score = state["score"]
    revision_count = state["revision_count"]
    if score >= 80:
        return {}

    return {
        "revision_count":
        revision_count + 1
    }

# --------------------------------------------------
# Routing
# --------------------------------------------------

def route_after_critic(
    state: SupervisorState
):
    if state["score"] >= 80:
        return "finalizer"
    
    if state["revision_count"] >= 2:
        return "finalizer"

    return "writer"


# --------------------------------------------------
# Finalizer
# --------------------------------------------------

def finalizer(state: SupervisorState):

    response = client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {
                "role": "system",
                "content":
                "최종 답변 작성 Agent"
            },
            {
                "role": "user",
                "content":
                f"""
작업:
{state['task']}

조사:
{state['research_notes']}

최종 초안:
{state['draft']}

비평:
{state['critique']}

최종 답변 작성
"""
            }
        ]
    )

    return {
        "final_answer":
        response.choices[0].message.content.strip()
    }


# --------------------------------------------------
# Graph
# --------------------------------------------------

workflow = StateGraph(
    SupervisorState
)

workflow.add_node(    "researcher",    researcher)
workflow.add_node(    "writer",    writer)
workflow.add_node(    "critic",    critic)
workflow.add_node(    "supervisor",    supervisor)
workflow.add_node(    "finalizer",    finalizer)
workflow.add_edge(    START,    "researcher")
workflow.add_edge(    "researcher",    "writer")
workflow.add_edge(    "writer",    "critic")
workflow.add_edge(    "critic",    "supervisor")
workflow.add_conditional_edges(
    "supervisor",
    route_after_critic,
    {
        "writer": "writer",
        "finalizer": "finalizer",
    }
)
workflow.add_edge(    "finalizer",    END)
app = workflow.compile()

# --------------------------------------------------
# Run
# --------------------------------------------------

result = app.invoke(
    {
        "task":
        "LangGraph Supervisor 기반 멀티 에이전트 패턴 설명",

        "research_notes": "",

        "draft": "",

        "critique": "",

        "score": 0,

        "revision_count": 0,

        "final_answer": "",
    }
)

print("\n=== FINAL ===")
print(result["final_answer"])

print("\n=== SCORE ===")
print(result["score"])

print("\n=== REVISION ===")
print(result["revision_count"])

print("\n=== CRITIQUE ===")
print(result["critique"])


=== FINAL ===
아래는 위 초안을 **“바로 구현 가능한 수준”**으로 끌어올리기 위해, LangGraph의 **상태 머지/조건부 라우팅(conditional edges)/Supervisor 종료 기준**까지 문서 흐름에 연결해 정리한 **개선 최종 버전**입니다. (핵심은: *상태 필드 정의 → 어떤 노드가 어떤 필드를 append/replace → 게이트 조건이 정확히 어떤 코드/규칙으로 라우팅* 되는 걸 한 번에 보이게 하는 것)

---

## 0) 전체 패턴 한 줄 요약
- **LangGraph(상태 기반 그래프)**: 모든 에이전트는 같은 `State`를 읽고/업데이트하며 협업
- **Supervisor 기반 멀티에이전트**: Supervisor가 `State`의 조건을 근거로 “다음 노드(에이전트)”를 결정하고 종료/재호출까지 관리
- **하위 에이전트(ReAct 등)**: 도구(tool) 결과를 **반드시 `State`의 특정 필드에 append/replace**로 기록 → 그게 다음 라우팅 게이트의 근거가 됨

---

## 1) 상태 스키마(State) + 머지 규칙(append/replace 명시)

### 1-1. State 필드(예시)
```python
from typing import TypedDict, List, Literal, Optional

class State(TypedDict):
    # input
    user_request: str
    task: str

    # pipeline artifacts
    research_notes: List[str]          # 누적
    claims: List[str]                  # 보통 누적 또는 단일 배치 replace(선택)
    citations: List[dict]             # 누적(후보 -> 검증 전/후 구분 가능)

    draft: Optional[str]
    final_answer: Optional[st

# [8교시]

https://docs.langchain.com/  참고해서 예시 코드 작성

In [ ]:
import os
import re
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from openai import OpenAI
from dotenv import load_dotenv

# API 키 로드
load_dotenv(override=True)
client = OpenAI()

# ==========================================
# 1. 상태(State) 설계
# 멀티 에이전트들이 서로 주고받을 '공유 메모리'입니다.
# ==========================================
class ExplanationState(TypedDict):
    question: str           # 틀린 문제 내용
    user_answer: str        # 사용자가 고른 오답
    correct_answer: str     # 실제 정답
    
    research_notes: str     # DB(ChromaDB)에서 찾은 관련 개념
    draft: str              # Writer가 작성한 해설 초안
    critique: str           # Critic이 평가한 비평 내용
    score: int              # 해설 품질 점수 (100점 만점)
    revision_count: int     # 수정 횟수 (무한 루프 방지용)
    
    final_explanation: str  # 최종 확정된 해설

# ==========================================
# 2. 에이전트 노드(Nodes) 정의
# ==========================================

# (1) Researcher: 문제와 관련된 핵심 개념을 수집하는 역할
def researcher(state: ExplanationState):
    print("Agent [Researcher]: 관련 개념을 검색 중입니다...")
    # [설명] 실제 환경에서는 여기에 3교시에 만드신 ChromaDB 검색 코드를 넣습니다.
    # result = collection.query(query_texts=[state["question"]], n_results=2)
    # context = "\n".join(result["documents"][0])
    
    # 지금은 테스트를 위해 가상의 검색 결과를 반환합니다.
    dummy_context = (
        "OSI 7계층 중 전송 계층(Transport Layer)은 종단 간(End-to-End) 신뢰성 있는 "
        "데이터 전송을 담당하며, 대표적인 프로토콜로 TCP와 UDP가 있다. "
        "데이터 링크 계층(Data Link Layer)은 인접한 노드 간의 데이터 전송을 담당한다."
    )
    
    return {"research_notes": dummy_context}

# (2) Writer: 수집된 정보를 바탕으로 해설 초안을 작성하는 역할
def writer(state: ExplanationState):
    print(f"Agent [Writer]: 해설을 작성 중입니다. (현재 수정 횟수: {state.get('revision_count', 0)})")
    
    # [설명] 비평(critique)이 있다면 프롬프트에 포함시켜 피드백을 반영하도록 합니다.
    feedback_prompt = f"이전 비평을 반드시 반영하여 수정하세요:\n{state.get('critique', '없음')}" if state.get('critique') else ""

    response = client.chat.completions.create(
        model="gpt-4o-mini", # 비용 효율을 위해 mini 사용
        messages=[
            {
                "role": "system",
                "content": "당신은 IT 자격증 1타 강사입니다. 비전공자도 이해하기 쉽게 친절하게 설명해 주세요."
            },
            {
                "role": "user",
                "content": f"""
다음 틀린 문제에 대한 해설을 작성하세요.
사용자가 왜 저 오답을 골랐는지 짚어주고, 정답인 이유를 '관련 개념'을 바탕으로 설명하세요.

- 문제: {state['question']}
- 사용자가 고른 오답: {state['user_answer']}
- 정답: {state['correct_answer']}
- 관련 개념: {state['research_notes']}

{feedback_prompt}
"""
            }
        ],
        temperature=0.7
    )
    
    return {"draft": response.choices[0].message.content.strip()}

# (3) Critic (비평가): Writer가 쓴 해설의 퀄리티를 평가하는 역할
def critic(state: ExplanationState):
    print("Agent [Critic]: 해설의 품질을 평가 중입니다...")
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": """
당신은 엄격한 교육 콘텐츠 검수자입니다.
해설이 정확한지, 오답을 고른 이유를 잘 설명했는지 평가하세요.
반드시 아래 형식으로만 답변하세요.
SCORE: [0~100 사이 숫자]
CRITIQUE: [구체적인 수정 요구사항 또는 칭찬]
"""
            },
            {
                "role": "user",
                "content": f"문제: {state['question']}\n초안:\n{state['draft']}"
            }
        ],
        temperature=0
    )
    
    result_text = response.choices[0].message.content.strip()
    
    # 정규식으로 SCORE 추출
    score = 0
    match = re.search(r"SCORE:\s*(\d+)", result_text)
    if match:
        score = int(match.group(1))
        
    return {"critique": result_text, "score": score}

# (4) Supervisor: 수정 횟수를 관리하고 종료 여부를 체크하는 관리자
def supervisor(state: ExplanationState):
    # [설명] 상태(State)의 revision_count를 1 증가시킵니다. (무한 루프 방지)
    current_count = state.get("revision_count", 0)
    return {"revision_count": current_count + 1}

# (5) Finalizer: 검증을 통과한 해설을 최종 정리하여 출력하는 역할
def finalizer(state: ExplanationState):
    print("Agent [Finalizer]: 최종 해설을 확정합니다. 🎉")
    # 초안을 그대로 최종본으로 쓰거나, 마크다운 등 UI에 맞게 꾸미는 작업을 할 수 있습니다.
    final_text = f"## 📝 오답 노트 해설\n\n{state['draft']}"
    return {"final_explanation": final_text}

# ==========================================
# 3. 라우팅 로직 (조건부 엣지)
# ==========================================
def route_after_critic(state: ExplanationState):
    # [설명] 비평 점수가 80점 이상이거나, 수정 횟수가 2번을 넘어가면 통과시킵니다.
    print(f"-> [라우팅 결정] 현재 점수: {state['score']}점")
    if state["score"] >= 80:
        return "finalizer"
    
    if state.get("revision_count", 0) >= 2:
        print("-> [라우팅 결정] 최대 수정 횟수 초과로 강제 종료(확정)합니다.")
        return "finalizer"

    print("-> [라우팅 결정] 점수 미달로 Writer에게 수정을 지시합니다.")
    return "writer"

# ==========================================
# 4. 그래프(Graph) 조립 및 컴파일
# ==========================================
workflow = StateGraph(ExplanationState)

# 노드 등록
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)
workflow.add_node("critic", critic)
workflow.add_node("supervisor", supervisor)
workflow.add_node("finalizer", finalizer)

# 엣지 연결 (흐름 정의)
workflow.add_edge(START, "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_edge("critic", "supervisor")

# Supervisor 노드가 끝난 뒤, 조건에 따라 갈림길 생성
workflow.add_conditional_edges(
    "supervisor",
    route_after_critic,
    {
        "writer": "writer",       # 재수정
        "finalizer": "finalizer"  # 최종 확정
    }
)

workflow.add_edge("finalizer", END)

# 앱 컴파일
app = workflow.compile()

# ==========================================
# 5. 실행 테스트
# ==========================================
if __name__ == "__main__":
    print("\n========== [오답 해설 생성기 시작] ==========\n")
    
    # 사용자가 스터디 플랫폼에 입력한 데이터(가정)
    inputs = {
        "question": "다음 중 종단 간 신뢰성 있는 전송을 보장하는 OSI 7계층은?",
        "user_answer": "데이터 링크 계층",
        "correct_answer": "전송 계층",
        "research_notes": "",
        "draft": "",
        "critique": "",
        "score": 0,
        "revision_count": 0,
        "final_explanation": ""
    }
    
    result = app.invoke(inputs)
    
    print("\n========== [최종 결과] ==========\n")
    print(result["final_explanation"])


========== [오답 해설 생성기 시작] ==========

Agent [Researcher]: 관련 개념을 검색 중입니다...
Agent [Writer]: 해설을 작성 중입니다. (현재 수정 횟수: 0)
Agent [Critic]: 해설의 품질을 평가 중입니다...
-> [라우팅 결정] 현재 점수: 90점
Agent [Finalizer]: 최종 해설을 확정합니다. 🎉

========== [최종 결과] ==========

## 📝 오답 노트 해설

### 문제 해설

사용자가 선택한 오답인 "데이터 링크 계층"은 OSI 7계층에서 데이터 전송과 관련된 중요한 역할을 하지만, 종단 간 신뢰성 있는 전송을 보장하는 계층이 아닙니다. 데이터 링크 계층은 주로 네트워크 내의 인접한 장비들 간에 데이터를 전송하는 책임을 가지고 있으며, 물리적인 연결을 통해 오류 감지 및 수정 같은 기능을 수행합니다. 그러나 종단 간 신뢰성을 보장하는 것과는 다릅니다.

### 정답 설명

정답인 "전송 계층"은 OSI 7계층의 네 번째 계층으로, 종단 간 신뢰성 있는 데이터 전송을 보장하는 역할을 합니다. 이 계층은 데이터가 출발지에서 목적지까지 안전하게 전달될 수 있도록 다양한 기능을 제공합니다. 

주요 프로토콜인 TCP(전송 제어 프로토콜)는 신뢰성 있는 전송을 보장하는 대표적인 프로토콜로, 데이터의 손실이나 순서 변경을 방지하기 위해 흐름 제어와 오류 검출 및 수정 기능을 갖추고 있습니다. 반면 UDP(사용자 데이터그램 프로토콜)는 신뢰성을 보장하지 않지만, 빠른 전송 속도가 필요한 경우 사용됩니다. 

따라서, 종단 간 신뢰성 있는 전송을 보장하는 기능은 전송 계층의 핵심적인 역할이며, 이로 인해 사용자가 오답을 선택한 이유는 데이터 링크 계층의 역할과 전송 계층의 역할을 혼동했기 때문일 수 있습니다. 데이터 링크 계층은 인접한 노드 간의 통신에 중점을 두고, 전송 계층은 출발지와 목적지 간의 신뢰성 있는 통신을 보장하는 데 초점을 맞추고 있습니다.
